# Inspect SEXTANTS NeXus metadata

Browse the complete structure, dataset metadata, attributes, and small values
inside one or more `.nxs` files. Large detector arrays are **not loaded** into
the inventory table. Use the search cell to find terms such as `ccd`, `energy`,
`polar`, `diode`, `field`, `motor`, or `data_03`.

In [ ]:
# Configure Qt before importing pyplot or any other GUI-aware library.
import os
import sys
from os.path import join

BASEFOLDER = os.path.abspath(os.getcwd())
LIBRARY_FOLDER = join(BASEFOLDER, "library")
if LIBRARY_FOLDER not in sys.path:
    sys.path.insert(0, LIBRARY_FOLDER)

from notebook_setup import configure_matplotlib_qt
MATPLOTLIB_BACKEND = configure_matplotlib_qt()
print("Matplotlib backend:", MATPLOTLIB_BACKEND)

import os, sys

from glob import glob
import h5py
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = os.path.abspath(os.getcwd())
sys.path.insert(0, join(ROOT, "library"))

from interactive import cimshow
from nexus_inspection import inspect_nexus, scalar_metadata, search_inventory

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_rows", 300)

## Choose raw folder and scans

In [ ]:
# Keep the beamline path for later; the second assignment is active for now.
RAW_FOLDER = "/nfs/ruche/sextants-soleil/com-sextants/COMET_20260902_Cocoons_Laser/"

# Set scan IDs explicitly, or use None to inspect every .nxs file in the folder.
SCAN_IDS = [13, 14]
FILENAME = "scanx_{scan_id:04d}.nxs"

if SCAN_IDS is None:
    files = sorted(glob(join(RAW_FOLDER, "*.nxs")))
else:
    files = [join(RAW_FOLDER, FILENAME.format(scan_id=scan_id)) for scan_id in SCAN_IDS]

missing = [path for path in files if not os.path.isfile(path)]
if missing:
    raise FileNotFoundError(f"Missing NeXus files: {missing}")
print(f"Inspecting {len(files)} file(s):")
for path in files:
    print(" ", path)

## Complete inventory

Each row is a group or dataset. `value` is shown only for small datasets;
`attributes` always includes all object attributes.

In [ ]:
inventories = {os.path.basename(path): inspect_nexus(path, max_preview_items=8) for path in files}
inventory = pd.DataFrame([row for rows in inventories.values() for row in rows])
display(inventory)
print("Objects:", len(inventory), "datasets:", int((inventory["kind"] == "dataset").sum()))

## Search paths, values, and attributes

In [ ]:
QUERY = "ccd"  # Examples: ccd-ts, data_03, energy, polar, diode, field, motor

matches = pd.DataFrame(search_inventory(inventory.to_dict("records"), QUERY))
display(matches)
print(f"{len(matches)} match(es) for {QUERY!r}")

## Commonly useful metadata

In [ ]:
TERMS = ["ccd-ts", "energy", "polar", "integration", "exposure", "diode", "field", "magnet"]
for filename, rows in inventories.items():
    print("\n", filename)
    useful = []
    seen = set()
    for term in TERMS:
        for row in search_inventory(rows, term):
            if row["path"] not in seen:
                useful.append(row)
                seen.add(row["path"])
    display(pd.DataFrame(useful))

## Compare all scalar metadata across scans

Rows are dataset paths and columns are files. This makes changing motor,
detector, energy, polarization, and timing values easy to spot.

In [ ]:
scalars = {os.path.basename(path): scalar_metadata(path) for path in files}
scalar_comparison = pd.DataFrame(scalars).sort_index()
display(scalar_comparison)

# Show only values that differ between the selected files.
if len(files) > 1:
    changing = scalar_comparison[
        scalar_comparison.apply(lambda row: len({repr(value) for value in row.dropna()}) > 1, axis=1)
    ]
    print("Changing scalar datasets:")
    display(changing)

## Inspect one dataset directly

Use a path copied from the inventory. Large arrays are summarized unless
`LOAD_FULL_DATASET` is deliberately enabled.

In [ ]:
FILE_INDEX = 0
DATASET_PATH = "/scan_0013/scan_data/data_03"
LOAD_FULL_DATASET = False

selected_file = files[FILE_INDEX]
with h5py.File(selected_file, "r") as handle:
    obj = handle[DATASET_PATH]
    print("File:", selected_file)
    print("Path:", obj.name)
    print("Type:", type(obj).__name__)
    print("Attributes:")
    for key, value in obj.attrs.items():
        print(f"  {key}: {value!r}")
    if isinstance(obj, h5py.Dataset):
        print("Shape:", obj.shape, "dtype:", obj.dtype, "size:", obj.size)
        if LOAD_FULL_DATASET or obj.size <= 100:
            value = obj[()]
            print("Value:", value)
        else:
            print("Value not loaded; set LOAD_FULL_DATASET=True if intentional.")

In [ ]:
# Acquisition ID summary
print("im_ids:", globals().get("IMAGE_IDS", globals().get("SCAN_IDS", None)))
print("dark_ids:", globals().get("DARK_IDS", None))